# Lesson 22 — Global Optimization Theory

## Learning objectives

1. Distinguish deterministic vs heuristic global optimization.
2. State convergence guarantees of spatial B&B and their hypotheses.
3. Recognize the role of factorable functions and convex envelopes.
4. Understand the Lasserre / SDP hierarchy for polynomial optimization.

## 1. Deterministic vs heuristic

- **Deterministic** (spatial B&B, αBB, BARON, Couenne, ANTIGONE): finite ε-convergence; certified optimum.
- **Heuristic** (DIRECT, GA, simulated annealing, basin hopping): no certified optimum; in practice often fast first incumbent but no termination criterion.

This course is about *deterministic* global optimization {cite:p}`Floudas1999,HorstTuy1996`. Heuristics have their place but cannot certify optimality.

## 2. Factorable functions

Most practical NLPs use **factorable** functions: built up from elementary operators (+, ×, ÷, exp, log, sin, ...) on continuous variables. The convex-envelope construction in Lesson 16 applies term-by-term.

For non-factorable functions (oracle-only, black-box), Bayesian optimization or trust-region surrogate methods are alternatives — but no global certificate.

## 3. Convergence

**Theorem (informal, {cite:p}`HorstTuy1996,Floudas1999`):** spatial B&B with consistent bounding (relaxation gap → 0 as box → point) and exhaustive partitioning ($\sup_N \text{diam}(B_N) \to 0$) converges to the global optimum.

The hypotheses are mild — McCormick + bisection satisfies them. So convergence isn't the question; *speed* is.

## 4. Worst-case complexity

**Unbounded-integer** polynomial MINLP is undecidable (Hilbert's 10th problem: deciding whether a polynomial Diophantine equation has an integer solution is undecidable, and it embeds into the feasibility question for an unbounded-integer NLP). For **bounded-box** problems with rational coefficients and bounded integer variables, the problem is decidable; the existential theory of the reals on bounded boxes is in PSPACE, and bounded MINLP is at least NP-hard already in the quadratic case. Finite-precision algorithms (including spatial B&B) are exponential in the worst case.

**Practical lesson:** in the worst case, spatial B&B is exponential in the number of nonconvex variables. Reformulate to convexify when possible (e.g., GDP, Lesson 23).

## 5. Lasserre / SDP hierarchy

For polynomial optimization

$$\min p(x) \;\;\text{s.t.}\;\; g_i(x) \ge 0,$$

a sequence of SDPs of increasing degree provides monotonically improving lower bounds, converging to the global optimum (Lasserre 2001). Practical for low-degree problems with few variables; doesn't scale to high-dimensional industrial problems.

Sum-of-squares (SOS) certificates are the dual viewpoint.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do
from discopt.modeling import Model

# Box-constrained polynomial; classical Goldstein-Price test function
m = Model("gp")
x = m.continuous("x", lb=-2, ub=2)
y = m.continuous("y", lb=-2, ub=2)
f = ((1 + (x + y + 1)**2 * (19 - 14*x + 3*x**2 - 14*y + 6*x*y + 3*y**2)) *
     (30 + (2*x - 3*y)**2 * (18 - 32*x + 12*x**2 + 48*y - 36*x*y + 27*y**2)))
m.minimize(f)

r = m.solve(time_limit=60)
# Known global min: f* = 3 at (0, -1)
print(f"status: {r.status}   incumbent f = {r.objective:.6f} at (x, y) = ({r.value(x):.4f}, {r.value(y):.4f})")
print(f"dual bound = {r.bound:.3f}   gap_certified = {r.gap_certified}")
# Expect: the incumbent is the global optimum almost immediately, but the
# degree-8 polynomial has very loose factorable relaxations, so within the
# time limit the bound is far from 3 and the status is "feasible", not
# "optimal". Finding the optimum and *proving* it are different jobs.


## 6. Certified vs uncertified answers in `discopt`

The deterministic/heuristic split of §1 is visible in every `discopt`
result. The status vocabulary is deliberately strict:

| `r.status` | Claim |
|---|---|
| `"optimal"` (with `r.gap_certified`) | Certified: valid dual bound, gap closed to tolerance. |
| `"feasible"` | A verified feasible point. If `r.bound_valid`, `r.bound` is a sound global bound but the gap is open (e.g. a time limit, as in the cell above). |
| `"local_optimal"` / `"local_infeasible"` | A *local* solver's verdict: a stationary point, or a failure to find one. No global claim either way. |
| `"infeasible"` | A *proof* of infeasibility (not just a failed search). |

Which engine produced the answer is recorded in `r.algorithm_route`.
The backends map onto §1 as follows:

| Call | Kind | Certificate? |
|---|---|---|
| `m.solve()` | spatial B&B (or a single NLP if the model is proved convex) | yes, for bounded factorable models |
| `m.solve(solver="amp")` | adaptive multivariate partitioning {cite:p}`Nagarajan2019` | yes |
| `m.solve(solver="mip-nlp", mip_nlp_method="oa")` | outer approximation and relatives | yes for convex MINLP only |
| `m.solve(solver="gp")` / `"gp-minlp"` | geometric programming in log space | yes (GP is convex after the transform) |
| `m.solve(solver="direct")` | DIRECT sampling search {cite:p}`Jones1993` | no — `bound`/`gap` are `None` |
| `m.solve(solver="surrogate")` | model-based search for expensive black boxes | no |
| `m.solve(skip_convex_check=True)` on a continuous model | one local NLP solve on the model as written (status `"feasible"`, `bound` is `None`) | no |
| `m.solve(solver="pounce")` | one interior-point solve, no tree (new in 0.10) | only for an LP or a QP proved PSD; otherwise `"local_optimal"` |
| `m.solve(solver="scip")` / `"gurobi"` | the whole model handed to an external solver (new in 0.10 for SCIP: `pip install "discopt[scip]"`) | no — see below |
| a model containing `dm.external` | a grey box with caller-supplied derivatives, solved by a local NLP (new in 0.10; Lesson 33) | no — `"feasible"`, `bound` is `None` |

`solver="direct"` and `"surrogate"` exist for models with opaque
`dm.custom` bodies that have no algebraic relaxation — the case §2 calls
non-factorable. As soon as one opaque term appears, the global certificate
is gone for the *whole* model.

**Whose certificate is it?** `solver="scip"` translates the model in memory
into PySCIPOpt expressions and lets SCIP, itself a deterministic global
solver, do the search. `discopt` still **never reports that answer as
certified**. A SCIP `optimal` comes back as `status="feasible"`, a SCIP
`infeasible` or `unbounded` as `status="unknown"`, always with
`gap_certified=False`, and SCIP's dual bound is labelled
`r.bound_source == "external_solver"` (compare `"lp_dual"` or
`"convex_proof"` on discopt's own routes). `discopt` re-verifies SCIP's point
on the original model, but it cannot re-check a bound it did not compute.
This is not distrust for its own sake: on the MINLPLib instance `ex1252a`,
SCIP's presolve reported a bound past the true optimum. A certificate is only
as good as the arithmetic behind it, and an external solver's arithmetic is
outside what `discopt` can audit. A function SCIP cannot represent exactly
(`atan`, `erf`, `min`/`max`, `dm.custom`) raises `ScipUnsupportedError`;
`discopt` never substitutes an approximation silently.

## References
{cite:p}`Floudas1999,HorstTuy1996,Tawarmalani2005,Belotti2013`.